# Kaggle — CodeT5 cache cho Devign validation và test

Chỉ embedding 2.732 validation + 2.732 test; không xử lý train, Joern, retrieval, demo hoặc prompt.

1. Upload `devign_codet5_kaggle_source.zip` thành Kaggle Dataset riêng và **Add Input** dataset đó vào notebook này. ZIP giữ nguyên hoặc Kaggle tự giải nén đều được.
2. Bật **GPU** và **Internet** trong Settings. Chạy lần lượt các cell.
3. Tải `devign_codet5_validation_test_cache.zip` ở Output sau khi hoàn tất.

Giữ đúng contract GRACE-summary: CodeT5-base revision ghim, T5Encoder, code `.strip()`, 512 token, masked mean, FP32. Cache chưa whitening. Batch mặc định 64, có thể đổi thành 32 nếu GPU hết VRAM. Cache lưu sau mỗi batch; chạy lại trong cùng `/kaggle/working` sẽ resume. Một GPU được sử dụng.


In [1]:
from pathlib import Path
import sys, subprocess, json, hashlib, zipfile, os, shutil
BATCH_SIZE = 64
WORK = Path('/kaggle/working/devign_codet5_job')
OUT = WORK / 'embedding_cache'
WORK.mkdir(parents=True, exist_ok=True)
subprocess.run(['nvidia-smi'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'transformers==4.57.1'], check=True)


Sat Oct 31 03:19:32 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.26.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.


CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'install', '-q', 'transformers==4.57.1'], returncode=0)

## Tìm bundle và kiểm tra toàn bộ mã nguồn/dữ liệu đầu vào

Cell tự tìm đúng tên bundle trong `/kaggle/input`. Nếu có nhiều bản, giữ lại đúng một input để tránh nhầm phiên bản.


In [2]:
INPUT = Path('/kaggle/input')
markers = list(INPUT.rglob('package_manifest.json'))
markers = [p for p in markers if p.parent.name == 'devign_codet5_kaggle']
if len(markers) == 1:
    SOURCE = markers[0].parent
else:
    bundles = list(INPUT.rglob('devign_codet5_kaggle_source.zip'))
    assert len(bundles) == 1, f'Cần đúng một bundle ZIP hoặc một thư mục nguồn; thấy {len(bundles)} ZIP, {len(markers)} manifest'
    extracted = WORK / 'source'
    extracted.mkdir(exist_ok=True)
    with zipfile.ZipFile(bundles[0]) as z:
        for item in z.infolist():
            target = (extracted / item.filename).resolve()
            assert target.is_relative_to(extracted.resolve()), 'Unsafe ZIP path'
        z.extractall(extracted)
    SOURCE = extracted / 'devign_codet5_kaggle'

def sha256(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda: f.read(8 * 1024 * 1024), b''): h.update(block)
    return h.hexdigest()

PACKAGE = json.loads((SOURCE / 'package_manifest.json').read_text())
for name, digest in PACKAGE['files'].items():
    assert sha256(SOURCE / name) == digest, f'Hash mismatch: {name}'
assert PACKAGE['splits'] == {'validation': 2732, 'test': 2732}
print('Verified source:', SOURCE)
print('Contract:', PACKAGE['contract_revision'])


Verified source: /kaggle/input/datasets/quangdung1111/devign-codet5-kaggle-source-zip/devign_codet5_kaggle
Contract: evaluation_contract_v3_context8192tokens_20261029


In [3]:
import torch
assert torch.cuda.is_available(), 'Bật GPU trong Kaggle Settings rồi chạy lại'
assert tuple(int(x) for x in torch.__version__.split('+')[0].split('.')[:2]) >= (2, 6), 'Cần PyTorch >=2.6 để nạp checkpoint pytorch_model.bin'
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM GiB:', round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1))
os.environ['HF_HOME'] = str(WORK / 'huggingface')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'


GPU: Tesla T4
VRAM GiB: 14.6


## Embedding validation và test

Không gọi script bundle/retrieval. Log hiện số mẫu hoàn tất, %, samples/s và ETA theo split. Nếu CUDA OOM, đổi `BATCH_SIZE=32` ở cell đầu rồi chạy lại cell này; các batch đã lưu không bị tính lại.


In [4]:
cmd = [sys.executable, '-u', str(SOURCE / 'prepare_embedding_cache.py'),
       '--raw-dir', str(SOURCE / 'raw'), '--out', str(OUT),
       '--splits', 'validation', 'test', '--batch-size', str(BATCH_SIZE)]
print('RUN:', ' '.join(cmd), flush=True)
subprocess.run(cmd, cwd=SOURCE, env=os.environ.copy(), check=True)


RUN: /usr/bin/python3 -u /kaggle/input/datasets/quangdung1111/devign-codet5-kaggle-source-zip/devign_codet5_kaggle/prepare_embedding_cache.py --raw-dir /kaggle/input/datasets/quangdung1111/devign-codet5-kaggle-source-zip/devign_codet5_kaggle/raw --out /kaggle/working/devign_codet5_job/embedding_cache --splits validation test --batch-size 64
Device: Tesla T4 | dtype: torch.float32
validation: 64/2732 (2.3%) | 7.35 samples/s | ETA 6.0 min
validation: 128/2732 (4.7%) | 11.45 samples/s | ETA 3.8 min
validation: 192/2732 (7.0%) | 13.86 samples/s | ETA 3.1 min
validation: 256/2732 (9.4%) | 15.76 samples/s | ETA 2.6 min
validation: 320/2732 (11.7%) | 17.17 samples/s | ETA 2.3 min
validation: 384/2732 (14.1%) | 18.22 samples/s | ETA 2.1 min
validation: 448/2732 (16.4%) | 19.06 samples/s | ETA 2.0 min
validation: 512/2732 (18.7%) | 19.80 samples/s | ETA 1.9 min
validation: 576/2732 (21.1%) | 20.37 samples/s | ETA 1.8 min
validation: 640/2732 (23.4%) | 20.78 samples/s | ETA 1.7 min
validation: 7

CompletedProcess(args=['/usr/bin/python3', '-u', '/kaggle/input/datasets/quangdung1111/devign-codet5-kaggle-source-zip/devign_codet5_kaggle/prepare_embedding_cache.py', '--raw-dir', '/kaggle/input/datasets/quangdung1111/devign-codet5-kaggle-source-zip/devign_codet5_kaggle/raw', '--out', '/kaggle/working/devign_codet5_job/embedding_cache', '--splits', 'validation', 'test', '--batch-size', '64'], returncode=0)

## Xác minh cache và đóng ZIP kết quả

ZIP có `validation/` và `test/` ở cấp đầu. Khi tải về, ghép hai thư mục này với `train/` của Devign đang embed trên máy. Không chép đè cache đang được tiến trình local ghi; chờ lượt local dừng hoặc hoàn tất train. Pipeline `CachedEmbedder` cần cả ba split trước khi retrieval.


In [5]:
import numpy as np
result = {'schema': 'devign_codet5_validation_test_export_v1',
          'contract_revision': PACKAGE['contract_revision'], 'model': PACKAGE['model'],
          'source_package_sha256': sha256(SOURCE / 'package_manifest.json'), 'files': {}}
for split in ('validation', 'test'):
    meta = json.loads((OUT / split / 'manifest.json').read_text())
    rows = [json.loads(l) for l in (SOURCE / 'raw' / (split + '.jsonl')).read_text().splitlines() if l.strip()]
    identity = [{'id': str(r['id']), 'code_sha256': hashlib.sha256(r['func'].strip().encode()).hexdigest()} for r in rows]
    assert meta['model'] == PACKAGE['model']
    assert meta['pooling'] == 'masked_mean' and meta['max_length'] == 512
    assert meta['rows'] == identity and meta['completed'] == len(identity) == 2732
    a = np.load(OUT / split / 'vectors.npy', mmap_mode='r', allow_pickle=False)
    assert a.shape == (2732, 768) and a.dtype == np.float32 and np.isfinite(a).all()
    for name in ('manifest.json', 'vectors.npy'):
        key = f'{split}/{name}'
        result['files'][key] = sha256(OUT / key)
    print(split, a.shape, 'verified')
(OUT / 'export_manifest.json').write_text(json.dumps(result, indent=2) + '\n')
RESULT_ZIP = Path('/kaggle/working/devign_codet5_validation_test_cache.zip')
with zipfile.ZipFile(RESULT_ZIP, 'w', compression=zipfile.ZIP_DEFLATED) as z:
    for split in ('validation', 'test'):
        for name in ('manifest.json', 'vectors.npy'):
            z.write(OUT / split / name, f'{split}/{name}')
    z.write(OUT / 'export_manifest.json', 'export_manifest.json')
print('Download:', RESULT_ZIP, '| MiB:', round(RESULT_ZIP.stat().st_size / 2**20, 2))
from IPython.display import FileLink, display
display(FileLink(str(RESULT_ZIP)))


validation (2732, 768) verified
test (2732, 768) verified
Download: /kaggle/working/devign_codet5_validation_test_cache.zip | MiB: 15.09


/kaggle/working/devign_codet5_validation_test_cache.zip